# 06_energy_controls: Energy Controls and Savings Optimization

This notebook demonstrates the estimation of HVAC energy savings under various setback strategies.
We use a discrete 2R2C thermal circuit model, calibrate it using historical data, integrate occupancy heat gains,
define occupant comfort setpoints, and run control simulations.

## Objectives
1. **Data Loading & Prep**: Load preprocessed heating data for a single device.
2. **2R2C Model Calibration**: Replicate physical modeling calibration using bounded L-BFGS-B with multi-start optimization.
3. **Occupancy Heat Gains**: Model internal sensible heat gains from occupant presence.
4. **Comfort Setpoints**: Define occupied comfort bounds and setback target setpoints.
5. **Baseline Scenario**: Simulate indoor temperatures and HVAC energy under a constant target setpoint.
6. **Setback Scenarios**: Evaluate Mild, Deep, and Weekend + Night setback strategies.
7. **Savings & Comfort Compliance**: Quantify estimated energy savings (kWh, %) and comfort compliance.
8. **Visualization**: Plot setpoint overlays, temperature responses, and energy consumption bar charts.


In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.optimize
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

# Resolve backend/notebooks directory for imports
sys.path.append(os.path.abspath('.'))
from utils import load_processed


## 1. Data Loading

We load the preprocessed heating data and select a single device for building thermal parameter calibration.


In [ ]:
# Load preprocessed heating data
df = load_processed('heating')
print("Loaded dataset. Shape:", df.shape)

# Select a single device for analysis
device_id = df['device_id'].dropna().unique()[0]
dev_df = df[df['device_id'] == device_id].copy()
dev_df.index = pd.to_datetime(dev_df.index)
print(f"Selected device: {device_id} with {len(dev_df)} rows")

# Extract required columns for calibration and simulation, drop NaNs
model_df = dev_df[[
    'status_temperature_in_celsius', 
    'outside_temperature', 
    'power_draw_kw', 
    'occupancy_flag', 
    'hour_of_day', 
    'is_weekend', 
    'target_temp'
]].rename(columns={
    'status_temperature_in_celsius': 'T_in',
    'outside_temperature': 'T_out',
    'power_draw_kw': 'Q',
    'target_temp': 'T_target_actual'
}).dropna()

print("Cleaned data shape:", model_df.shape)
model_df.head()


## 2. 2R2C Model Calibration

We define a discrete 2R2C thermal circuit model and calibrate its parameters ($R_{in}, C_{in}, R_{out}, C_{wall}$) using bounded L-BFGS-B optimization with 3 random restarts.


In [ ]:
# 2R2C Simulation function
def simulate_2r2c(params, T_in_init, T_wall_init, T_out, Q, dt):
    """Simulate indoor temperature for a discrete 2R2C model."""
    R_in, C_in, R_out, C_wall = params
    if np.any(np.asarray(params, dtype=float) <= 0):
        raise ValueError("2R2C parameters must all be positive.")

    dt_hours = dt / 3600.0
    n_steps = len(T_out)
    T_pred = np.empty(n_steps, dtype=float)
    T_indoor = float(T_in_init)
    T_wall = float(T_wall_init)
    T_pred[0] = T_indoor

    for k in range(n_steps - 1):
        indoor_delta = dt_hours / C_in * (((T_wall - T_indoor) / R_in) + Q[k])
        wall_delta = dt_hours / C_wall * (
            ((T_out[k] - T_wall) / R_out) - ((T_wall - T_indoor) / R_in)
        )
        T_indoor = T_indoor + indoor_delta
        T_wall = T_wall + wall_delta

        if not np.isfinite(T_indoor) or not np.isfinite(T_wall):
            T_pred[k + 1 :] = np.nan
            break

        if abs(T_indoor) > 1000.0 or abs(T_wall) > 1000.0:
            T_pred[k + 1 :] = np.nan
            break

        T_pred[k + 1] = T_indoor

    return T_pred

# Calibration settings (first 300 rows like S04 calibration)
calibration_df = model_df.iloc[:300].copy()
T_in_cal = calibration_df['T_in'].to_numpy(dtype=float)
T_out_cal = calibration_df['T_out'].to_numpy(dtype=float)
Q_cal = calibration_df['Q'].to_numpy(dtype=float)
dt = 900.0 # 15-minute interval
T_wall_init = (T_in_cal[0] + T_out_cal[0]) / 2.0
bounds = [(0.001, 50.0)] * 4

def objective(params):
    T_pred = simulate_2r2c(params, T_in_cal[0], T_wall_init, T_out_cal, Q_cal, dt)
    if not np.all(np.isfinite(T_pred)):
        return 1e12
    return float(np.mean((T_pred - T_in_cal) ** 2))

# Multi-restart L-BFGS-B calibration
np.random.seed(42)
restart_results = []
for i in range(3):
    x0 = np.random.uniform(0.1, 25.0, size=4)
    result = scipy.optimize.minimize(objective, x0, method='L-BFGS-B', bounds=bounds, options={'maxiter': 100})
    restart_results.append(result)

best_result = min(restart_results, key=lambda r: r.fun)
best_params = best_result.x
R_in, C_in, R_out, C_wall = best_params

print("Best Parameters:")
print(f"  R_in:   {R_in:.4f} °C/kW")
print(f"  C_in:   {C_in:.4f} kWh/°C")
print(f"  R_out:  {R_out:.4f} °C/kW")
print(f"  C_wall: {C_wall:.4f} kWh/°C")
print(f"Calibration MSE: {best_result.fun:.6f}")
print(f"Calibration RMSE: {np.sqrt(best_result.fun):.6f} °C")


## 3. Occupancy Heat Gains

We define a simple occupancy schedule and calculate sensible heat gains:
- **Occupancy Schedule**: weekdays 07:00–22:00, weekends 09:00–22:00.
- **Occupant Heat Gain Assumptions**: 2 occupants with sensible heat gain of 75W (0.075 kW) and latent heat gain of 55W (0.055 kW) per person.
- **Sensible Heat Gain**: $Q_{occ} = N_{people} \times q_{sensible} = 2 \times 0.075 = 0.15$ kW.


In [ ]:
# Define occupancy heat gain parameters
N_people = 2
q_sensible = 0.075 # kW per person
q_latent = 0.055 # kW per person

Q_occ_sensible = N_people * q_sensible
Q_occ_latent = N_people * q_latent
print(f"Occupancy Sensible Heat Gain: {Q_occ_sensible:.3f} kW")
print(f"Occupancy Latent Heat Gain: {Q_occ_latent:.3f} kW")

# Generate occupancy schedule
hour_of_day = model_df['hour_of_day'].to_numpy()
is_weekend = model_df['is_weekend'].to_numpy()
n_steps = len(model_df)

occupied = np.zeros(n_steps, dtype=bool)
for k in range(n_steps):
    hr = hour_of_day[k]
    if is_weekend[k] == 1:
        occupied[k] = (hr >= 9) and (hr < 22)
    else:
        occupied[k] = (hr >= 7) and (hr < 22)

# Save occupancy flags and heat gains to model_df
model_df['occupied_schedule'] = occupied
model_df['Q_occ'] = np.where(occupied, Q_occ_sensible, 0.0)

print(f"Occupied time steps: {occupied.sum()} out of {n_steps} ({occupied.sum()/n_steps*100:.1f}%)")


## 4. Comfort Setpoints and Simulation Logic

We define occupant comfort setpoints:
- **Occupied comfort bounds**: $T_{min\_occ} = 20.0^\circ$C, $T_{max\_occ} = 24.0^\circ$C.
- **Unoccupied setback temperature**: $T_{setback} = 16.0^\circ$C.
- **Max HVAC heating capacity**: $Q_{max}$ is estimated from the maximum historical power draw.


In [ ]:
# Comfort setpoint assumptions
T_min_occ = 20.0
T_max_occ = 24.0
T_setback = 16.0

Q_max = float(model_df['Q'].max())
print(f"Max HVAC heating capacity (Q_max): {Q_max:.2f} kW")

def simulate_thermostat(params, T_in_init, T_wall_init, T_out, T_setpoints, Q_occ, dt, Q_max):
    """Simulate thermal response and required HVAC power to maintain setpoints."""
    R_in, C_in, R_out, C_wall = params
    dt_hours = dt / 3600.0
    n_steps = len(T_out)
    
    T_indoor = np.empty(n_steps)
    T_wall = np.empty(n_steps)
    Q_hvac = np.empty(n_steps)
    
    T_indoor[0] = T_in_init
    T_wall[0] = T_wall_init
    
    for k in range(n_steps - 1):
        T_target_next = T_setpoints[k+1]
        
        # Required heat input to reach target
        Q_req = (T_target_next - T_indoor[k]) * C_in / dt_hours - (T_wall[k] - T_indoor[k]) / R_in - Q_occ[k]
        
        # Bounded by 0 (cannot cool) and Q_max (max heating capacity)
        Q_hvac[k] = np.clip(Q_req, 0.0, Q_max)
        
        # Propagate temperatures
        indoor_delta = dt_hours / C_in * (((T_wall[k] - T_indoor[k]) / R_in) + Q_hvac[k] + Q_occ[k])
        T_indoor[k+1] = T_indoor[k] + indoor_delta
        
        wall_delta = dt_hours / C_wall * (((T_out[k] - T_wall[k]) / R_out) - ((T_wall[k] - T_indoor[k]) / R_in))
        T_wall[k+1] = T_wall[k] + wall_delta
        
    Q_hvac[-1] = 0.0
    return T_indoor, T_wall, Q_hvac


## 5. Baseline Scenario

The baseline scenario maintains a constant indoor temperature target of $21.0^\circ$C.


In [ ]:
# Define constant setpoint target
T_target_baseline = np.full(n_steps, 21.0)
T_out_arr = model_df['T_out'].to_numpy()
Q_occ_arr = model_df['Q_occ'].to_numpy()
T_in_init = model_df['T_in'].iloc[0]
T_wall_init = (T_in_init + T_out_arr[0]) / 2.0

T_in_b, T_w_b, Q_hvac_b = simulate_thermostat(
    best_params, T_in_init, T_wall_init, T_out_arr, T_target_baseline, Q_occ_arr, dt, Q_max
)

baseline_kwh = float(np.sum(Q_hvac_b) * 0.25)
print(f"Simulated Baseline Energy Consumption: {baseline_kwh:.2f} kWh")

# Save baseline outcomes
model_df['T_in_baseline'] = T_in_b
model_df['Q_hvac_baseline'] = Q_hvac_b
model_df['T_target_baseline'] = T_target_baseline


## 6. Setback Scenarios

We evaluate three setback scenarios:
1. **Mild Night Setback**: Target setpoint is $18.0^\circ$C during unoccupied hours ($22:00$–$07:00$).
2. **Deep Night Setback**: Target setpoint is $16.0^\circ$C during unoccupied hours ($22:00$–$07:00$).
3. **Weekend + Night Setback**: Target setpoint is $16.0^\circ$C during unoccupied weekday hours, $21.0^\circ$C during occupied weekday hours, and $17.0^\circ$C constant on weekend days.


In [ ]:
# Construct setpoint arrays
T_target_mild = np.where(occupied, 21.0, 18.0)
T_target_deep = np.where(occupied, 21.0, 16.0)

T_target_wknd = np.empty(n_steps)
for k in range(n_steps):
    if is_weekend[k] == 1:
        T_target_wknd[k] = 17.0
    else:
        T_target_wknd[k] = 21.0 if occupied[k] else 16.0

# Run thermostat simulations
T_in_mild, T_w_mild, Q_hvac_mild = simulate_thermostat(
    best_params, T_in_init, T_wall_init, T_out_arr, T_target_mild, Q_occ_arr, dt, Q_max
)

T_in_deep, T_w_deep, Q_hvac_deep = simulate_thermostat(
    best_params, T_in_init, T_wall_init, T_out_arr, T_target_deep, Q_occ_arr, dt, Q_max
)

T_in_wknd, T_w_wknd, Q_hvac_wknd = simulate_thermostat(
    best_params, T_in_init, T_wall_init, T_out_arr, T_target_wknd, Q_occ_arr, dt, Q_max
)

# Store setback outcomes
model_df['T_in_mild'] = T_in_mild
model_df['Q_hvac_mild'] = Q_hvac_mild
model_df['T_target_mild'] = T_target_mild

model_df['T_in_deep'] = T_in_deep
model_df['Q_hvac_deep'] = Q_hvac_deep
model_df['T_target_deep'] = T_target_deep

model_df['T_in_wknd'] = T_in_wknd
model_df['Q_hvac_wknd'] = Q_hvac_wknd
model_df['T_target_wknd'] = T_target_wknd


## 7. Savings and Comfort Compliance Analysis

For each scenario, we calculate the total energy consumed (kWh), absolute savings (kWh), relative savings (%),
and count comfort violations (number of occupied 15-min intervals where $T_{in} < T_{min\_occ}$).


In [ ]:
scenarios = {
    'Baseline (Constant 21°C)': {
        'kWh': np.sum(Q_hvac_b) * 0.25,
        'T_in_sim': T_in_b
    },
    'Mild Night Setback (18°C/21°C)': {
        'kWh': np.sum(Q_hvac_mild) * 0.25,
        'T_in_sim': T_in_mild
    },
    'Deep Night Setback (16°C/21°C)': {
        'kWh': np.sum(Q_hvac_deep) * 0.25,
        'T_in_sim': T_in_deep
    },
    'Weekend + Night Setback (16°C/17°C/21°C)': {
        'kWh': np.sum(Q_hvac_wknd) * 0.25,
        'T_in_sim': T_in_wknd
    }
}

comparison_rows = []
for name, data in scenarios.items():
    kwh = float(data['kWh'])
    savings_kwh = baseline_kwh - kwh
    savings_pct = (savings_kwh / baseline_kwh) * 100.0 if baseline_kwh > 0 else 0.0
    
    t_in_sim = data['T_in_sim']
    violations = int(np.sum((t_in_sim < T_min_occ) & model_df['occupied_schedule']))
    
    comfort_status = "Compliant" if violations == 0 else f"Non-Compliant ({violations} violations)"
    
    comparison_rows.append({
        'Scenario': name,
        'HVAC Energy (kWh)': round(kwh, 2),
        'Savings (kWh)': round(savings_kwh, 2),
        'Savings (%)': round(savings_pct, 2),
        'Occupied Comfort Violations': violations,
        'Comfort Status': comfort_status
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


## 8. Visualization

We plot setpoint schedules, temperature responses, and energy savings comparisons.


In [ ]:
# Plot 1: Setpoint Schedule Overlay (48-hour subset)
plt.figure(figsize=(12, 5))
plot_subset = model_df.iloc[:192]

plt.plot(plot_subset.index, plot_subset['T_target_baseline'], label='Baseline (Constant 21°C)', linestyle='--', color='gray')
plt.plot(plot_subset.index, plot_subset['T_target_mild'], label='Mild Night Setback', color='blue')
plt.plot(plot_subset.index, plot_subset['T_target_deep'], label='Deep Night Setback', color='green')
plt.plot(plot_subset.index, plot_subset['T_target_wknd'], label='Weekend + Night Setback', color='orange')

plt.title('Thermostat Setpoint Schedule Overlay (48-Hour Sample)')
plt.ylabel('Setpoint Temperature (°C)')
plt.xlabel('Timestamp')
plt.grid(True, alpha=0.3)
plt.legend(loc='lower left')
plt.tight_layout()
plt.show()

# Plot 2: Simulated Temperature Response with Comfort Bounds Shaded
plt.figure(figsize=(12, 6))
plot_subset_4d = model_df.iloc[:384]

plt.plot(plot_subset_4d.index, plot_subset_4d['T_in_baseline'], label='Baseline Sim T_in', linestyle='--', color='gray')
plt.plot(plot_subset_4d.index, plot_subset_4d['T_in_mild'], label='Mild Setback Sim T_in', color='blue')
plt.plot(plot_subset_4d.index, plot_subset_4d['T_in_deep'], label='Deep Setback Sim T_in', color='green')
plt.plot(plot_subset_4d.index, plot_subset_4d['T_in_wknd'], label='Weekend+Night Sim T_in', color='orange')
plt.plot(plot_subset_4d.index, plot_subset_4d['T_out'], label='Outdoor Temp (T_out)', color='red', alpha=0.5, linewidth=1)

occupied_subset = plot_subset_4d['occupied_schedule'].to_numpy()
time_vals = plot_subset_4d.index
plt.fill_between(time_vals, T_min_occ, T_max_occ, where=occupied_subset, color='lightgreen', alpha=0.15, label='Occupied Comfort Zone (20°C - 24°C)')

plt.title('Simulated Indoor Temperature Response vs Comfort Bounds')
plt.ylabel('Temperature (°C)')
plt.xlabel('Timestamp')
plt.grid(True, alpha=0.3)
plt.legend(loc='lower left')
plt.tight_layout()
plt.show()

# Plot 3: Bar Chart of HVAC energy by scenario with savings annotated
plt.figure(figsize=(10, 6))
bars = plt.bar(comparison_df['Scenario'], comparison_df['HVAC Energy (kWh)'], color=['gray', 'blue', 'green', 'orange'])
plt.ylabel('HVAC Energy (kWh)')
plt.title('HVAC Energy Consumption and Savings by Scenario')
plt.xticks(rotation=15, ha='right')

# Annotate savings
for bar, row in zip(bars, comparison_rows):
    yval = bar.get_height()
    pct = row['Savings (%)']
    if pct > 0:
        plt.text(bar.get_x() + bar.get_width()/2.0, yval + 10, f"-{pct:.1f}% ({row['Savings (kWh)']:.0f} kWh)", ha='center', va='bottom', fontweight='bold')
    else:
        plt.text(bar.get_x() + bar.get_width()/2.0, yval + 10, "Baseline", ha='center', va='bottom')

plt.ylim(0, max(comparison_df['HVAC Energy (kWh)']) * 1.15)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## 9. Caveats and Limitations

- **Simplified 2R2C Model**: The thermal parameter estimation is based on a lumped 2R2C thermal circuit model. Actual buildings exhibit more complex multi-zone thermal dynamics and secondary heat transfer mechanisms.
- **Actuator Behavior & Cycling**: The thermostat control simulation assumes idealized, continuous modulation of heat input. Real systems operate with discrete compressor/cycling stages, hysteresis loops, or minimum runtime constraints.
- **Weather & Solar Gains**: Direct solar irradiance, wind, and radiative gains are not explicitly modeled here, but are lumped into the thermal resistance parameters.
- **Occupancy & Behavior Assumptions**: The occupancy schedule is assumed deterministic with rigid hours. Real occupant patterns are stochastic and lead to dynamic internal gains.
- **Dynamic Heating Capacity**: HVAC heating capacity is capped at historical maximum power draw. Actual heating capacity depends dynamically on outdoor temperature (e.g. COP degradation at sub-zero temperatures).
